# Voice Typing Classifier — speaker-independent nested CV

# Library Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, balanced_accuracy_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Function to construct average confusion matrices

In [2]:
def plot_mean_confumat(cm_avr, cm_sd, ax, title, clas_lab, set_cbar={'cmap':'viridis','cbar_shrink':1,'cmax':None,'cmin':None}, fontsz = 12):
  if set_cbar['cmin'] or set_cbar['cmax']:
    cmap_min, cmap_max = set_cbar['cmin'], set_cbar['cmax']
  else:
    cmap_min, cmap_max = cm_avr.min(), cm_avr.max()

  im=ax.imshow(cm_avr,vmin=cmap_min, vmax=cmap_max,interpolation='nearest',cmap=set_cbar['cmap'])

  cc_min, cc_max = im.cmap(0), im.cmap(256)
  cbar = plt.colorbar(im,ax=ax,shrink=set_cbar['cbar_shrink'])
  cbar.ax.tick_params(labelsize=fontsz)

  thresh = (cmap_max + cmap_min) / 2.0
  n_classes = cm_avr.shape[0]
  for r,c in product(np.arange(n_classes),np.arange(n_classes)):
      color = cc_max if cm_avr[r, c] < thresh else cc_min
      text = ax.text(c,r,'{mean:.2f}\n ({std:.2f})'.format(mean=cm_avr[r,c],
                      std=cm_sd[r,c]),ha="center",va="center",color=color, fontsize = fontsz)
  ax.set(xticks=np.arange(n_classes), yticks=np.arange(n_classes),
         xticklabels=clas_lab, yticklabels=clas_lab,
         ylabel="True label", xlabel="Predicted label")
  for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] +
             ax.get_xticklabels() + ax.get_yticklabels()):
    item.set_fontsize(fontsz)
  ax.set_title(title,{'fontsize':fontsz})
  plt.tight_layout()

# Data Loading

In [3]:
if not os.path.exists('voice_waveshape'):
  !git clone https://github.com/joaquinr-uner/voice_waveshape.git

Error loading required libraries. If there is an ongoing installation please wait for it to complete. Otherwise reinstall. (dlopen(@rpath/libxcodebuildLoader.dylib, 0x0001): Symbol not found: __ZN38pxrInternal_v0_21__aapl__pxrReserved__11TfSingletonINS_15TfDiagnosticMgrEE15_CreateInstanceEv
  Referenced from: <718775DA-E2F8-3DF0-8F50-8738F68B8C24> /Applications/Xcode.app/Contents/SystemFrameworks/USDKit.framework/Versions/A/USDKit
  Expected as weak-def export from some loaded dylib)
git: error: unable to locate xcodebuild, please make sure the path to the Xcode folder is set correctly!
git: error: You can set the path to the Xcode folder using /usr/bin/xcode-select -switch


In [4]:
CSV_PATH = 'voice_waveshape/Features_VoiceTyping.csv'
df_full = pd.read_csv(CSV_PATH)

# Unique speaker ID across databases (both databases number patients from 1)
df_full['Speaker'] = df_full['Database'] + '_' + df_full['Patient'].astype(str)

# Features: all except labels/IDs, and excluding TV_B1 and TV_F0 (as in the original notebook)
FEATURES = [c for c in df_full.columns
            if c not in ('Type', 'Database', 'Patient', 'Speaker', 'TV_B1', 'TV_F0')]
print(len(FEATURES), 'features')
print(df_full.groupby('Database').agg(signals=('Speaker', 'size'), speakers=('Speaker', 'nunique')))

FileNotFoundError: [Errno 2] No such file or directory: 'voice_waveshape/Features_VoiceTyping.csv'

# Settings
Hyperparameter grid (same as the original notebook):

*   $C_{Lin}$: $10^{-2} - 10^{2}$, $L = 7$
*   $C_{RBF}$: $10^{-1} - 10^{3}$, $L = 7$
*   $\gamma$: $\frac{0.01}{N_{features}} - \frac{100}{N_{features}}$, $L = 7$

Cost: `N_OUTER x N_INNER x 343` cascade fits per database. `N_JOBS = -1` runs the grid in parallel on all cores.

In [ ]:
N_OUTER = 10
N_INNER = 5
SEED = 0
N_JOBS = -1

VCLin = np.logspace(-2, 2, num=7)
VCRBF = np.logspace(-1, 3, num=7)
Vgamm = 1/len(FEATURES)*np.logspace(-2, 2, num=7)
GRID = list(product(VCLin, VCRBF, Vgamm))
len(GRID)

RESULTS_DIR = 'results/VoiceTyping'   # CSV tables and figures are written here
os.makedirs(RESULTS_DIR, exist_ok=True)

# Cascade SVM (Linear SVM + RBF SVM) with nested CV
Stage 1: linear SVM separates {1+2} vs {3}. Stage 2: RBF SVM, trained on classes 1 and 2 only, labels the samples stage 1 predicted as {1+2}.

In [ ]:
def fit_predict_cascade(Xtr, Y3tr, Xte, C1, C2, gam, return_score=False):
  # Scaler fitted on the training part only
  sc = StandardScaler().fit(Xtr)
  Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
  Ytr = (Y3tr == 3).astype(int)
  net1 = SVC(C=C1, kernel='linear', class_weight='balanced').fit(Xtr, Ytr)
  y_pred = net1.predict(Xte)
  net2 = SVC(C=C2, gamma=gam, class_weight='balanced').fit(Xtr[Ytr == 0], Y3tr[Ytr == 0])
  y3_pred = np.full(len(Xte), 3)
  if (y_pred == 0).any():
    y3_pred[y_pred == 0] = net2.predict(Xte[y_pred == 0])
  if return_score:
    # Stage-1 decision function: continuous score for type 3 vs types 1+2 (used for AUC)
    return y_pred, y3_pred, net1.decision_function(Xte)
  return y_pred, y3_pred


def inner_score(X, Y3, folds, params):
  # Mean 3-class balanced accuracy of one hyperparameter combination over the inner folds
  s = []
  for itr, iva in folds:
    _, y3p = fit_predict_cascade(X[itr], Y3[itr], X[iva], *params)
    s.append(balanced_accuracy_score(Y3[iva], y3p))
  return np.mean(s)


def metrics_bin(y, y_pred, score):
  # Binary task (type 3 = positive): balanced accuracy, sensitivity, specificity, AUC of the stage-1 score
  return {'BA': balanced_accuracy_score(y, y_pred), 'Sens': recall_score(y, y_pred, pos_label=1),
          'Spec': recall_score(y, y_pred, pos_label=0), 'AUC': roc_auc_score(y, score)}


def metrics_multi(y3, y3_pred):
  # 3-class task: balanced accuracy, per-type sensitivity (recall) and one-vs-rest specificity
  out = {'BA': balanced_accuracy_score(y3, y3_pred)}
  for c in (1, 2, 3):
    out[f'Sens {c}'] = recall_score(y3 == c, y3_pred == c)
    out[f'Spec {c}'] = recall_score(y3 != c, y3_pred != c)
  out['Spec (macro)'] = np.mean([out[f'Spec {c}'] for c in (1, 2, 3)])
  return out


def fmt_metrics(m):
  # DataFrame of per-fold metrics -> 'mean ± sd' per column (population sd, as np.std elsewhere)
  return pd.Series({c: f'{m[c].mean():.3f} ± {m[c].std(ddof=0):.3f}' for c in m.columns})


def nested_cv(df, grid=GRID, n_outer=N_OUTER, n_inner=N_INNER, seed=SEED, n_jobs=N_JOBS, verbose=True):
  X = df[FEATURES].to_numpy()
  Y3 = df['Type'].to_numpy()
  Y = (Y3 == 3).astype(int)
  g = df['Speaker'].to_numpy()

  outer = StratifiedGroupKFold(n_splits=n_outer, shuffle=True, random_state=seed)
  res = {'scores_bin': [], 'scores_multi': [], 'confumat2': [], 'confumat3': [],
         'chosen': [], 'inner_best': [], 'm_bin': [], 'm_multi': []}
  for k, (Tr, Te) in enumerate(outer.split(X, Y3, g)):
    assert not set(g[Tr]) & set(g[Te]), 'speaker in both outer train and test'

    # Inner grouped CV on the outer training fold only; same folds for every combination
    inner = StratifiedGroupKFold(n_splits=n_inner, shuffle=True, random_state=seed)
    folds = list(inner.split(X[Tr], Y3[Tr], g[Tr]))
    inner_means = Parallel(n_jobs=n_jobs)(
        delayed(inner_score)(X[Tr], Y3[Tr], folds, p) for p in grid)
    best = grid[int(np.argmax(inner_means))]

    # Refit on the whole outer training fold and score once on the outer test fold
    y_pred, y3_pred, score = fit_predict_cascade(X[Tr], Y3[Tr], X[Te], *best, return_score=True)
    res['m_bin'].append(metrics_bin(Y[Te], y_pred, score))
    res['m_multi'].append(metrics_multi(Y3[Te], y3_pred))
    res['scores_bin'].append(balanced_accuracy_score(Y[Te], y_pred))
    res['scores_multi'].append(balanced_accuracy_score(Y3[Te], y3_pred))
    res['confumat2'].append(confusion_matrix(Y[Te], y_pred, labels=[0, 1], normalize='true'))
    res['confumat3'].append(confusion_matrix(Y3[Te], y3_pred, labels=[1, 2, 3], normalize='true'))
    res['chosen'].append(dict(zip(['CLin', 'C_RBF', 'gamma'], best)))
    res['inner_best'].append(np.max(inner_means))
    if verbose:
      print(f'fold {k+1}/{n_outer}: test speakers={df["Speaker"].iloc[Te].nunique()}, inner best={np.max(inner_means):.3f}, '
            f'outer bin={res["scores_bin"][-1]:.3f}, outer multi={res["scores_multi"][-1]:.3f}, params={best}')
  for key in ('scores_bin', 'scores_multi', 'confumat2', 'confumat3', 'inner_best'):
    res[key] = np.array(res[key])
  for key in ('m_bin', 'm_multi'):
    res[key] = pd.DataFrame(res[key])   # one row per outer fold
  return res


def plot_results(res, DB):
  fig, ax = plt.subplots(1,2,figsize=(21,21))
  cbars = 0.3
  fntsz = 24
  cm2_avr, cm2_sd = res['confumat2'].mean(axis=0), res['confumat2'].std(axis=0)
  cm3_avr, cm3_sd = res['confumat3'].mean(axis=0), res['confumat3'].std(axis=0)

  title2 = f'SVM Lin. DB: {DB} (nested CV)' + '\n' + f'Average Balanced Accuracy (Std in brackets): {round(np.mean(res["scores_bin"]),2)} ({round(np.std(res["scores_bin"]),2)})'
  plot_mean_confumat(cm2_avr, cm2_sd, ax[0], title2, ['1+2','3'], set_cbar = {'cmap':'pink','cbar_shrink':cbars, 'cmin':None, 'cmax':None}, fontsz = fntsz)

  title3 = f'SVM Lin+RBF. DB: {DB} (nested CV)' + '\n' + f'Average Balanced Accuracy (Std in brackets): {round(np.mean(res["scores_multi"]),2)} ({round(np.std(res["scores_multi"]),2)})'
  plot_mean_confumat(cm3_avr, cm3_sd, ax[1], title3, ['1','2','3'],set_cbar = {'cmap':'pink','cbar_shrink':cbars, 'cmin':None, 'cmax':None}, fontsz = fntsz)
  plt.savefig(os.path.join(RESULTS_DIR, f'confumat_{DB}.png'), dpi=150, bbox_inches='tight')

# Training-Validation with full database (Meei+Saar)

In [ ]:
results = {}
DB = 'All'
df = df_full
results[DB] = nested_cv(df)

In [ ]:
pd.DataFrame(results[DB]['chosen']).assign(inner_best=results[DB]['inner_best'], outer_multi=results[DB]['scores_multi'])

## Plotting

In [ ]:
plot_results(results[DB], DB)

# Training-Validation with Meei database

In [ ]:
DB = 'Meei'
df = df_full[df_full['Database'] == 'Meei'].reset_index(drop=True)
results[DB] = nested_cv(df)

In [ ]:
pd.DataFrame(results[DB]['chosen']).assign(inner_best=results[DB]['inner_best'], outer_multi=results[DB]['scores_multi'])

## Plotting

In [ ]:
plot_results(results[DB], DB)

# Training-Validation with Saar database

In [ ]:
DB = 'Saar'
df = df_full[df_full['Database'] == 'Saar'].reset_index(drop=True)
results[DB] = nested_cv(df)

In [ ]:
pd.DataFrame(results[DB]['chosen']).assign(inner_best=results[DB]['inner_best'], outer_multi=results[DB]['scores_multi'])

## Plotting

In [ ]:
plot_results(results[DB], DB)

# Summary
`inner_best` is the tuned (optimistic) inner score; `outer` is the unbiased nested estimate.

In [ ]:
pd.DataFrame({DB: {'outer bin (mean)': r['scores_bin'].mean(), 'outer bin (sd)': r['scores_bin'].std(),
                   'outer multi (mean)': r['scores_multi'].mean(), 'outer multi (sd)': r['scores_multi'].std(),
                   'inner best multi (mean)': r['inner_best'].mean()}
              for DB, r in results.items()}).round(3)

## Detailed metrics (speaker-independent nested CV)
Mean ± sd over the 10 outer folds. Binary task: type 3 is the positive class; AUC uses the stage-1 linear SVM decision function. 3-class task: per-type sensitivity (recall) and one-vs-rest specificity; the mean of the three sensitivities equals the balanced accuracy. No 3-class AUC is given because the cascade has no continuous 3-class score.

In [ ]:
detail_table = pd.DataFrame({DB: {**{('Binary', m): v for m, v in fmt_metrics(r['m_bin']).items()},
                                  **{('3-class', m): v for m, v in fmt_metrics(r['m_multi']).items()}}
                             for DB, r in results.items()})
detail_table

# Save results
Writes the detailed metrics, every outer fold's selected hyperparameters and metrics, and a pickle of the result.

In [ ]:
detail_table.to_csv(os.path.join(RESULTS_DIR, 'detailed_metrics.csv'), encoding='utf-8-sig')

per_fold = []
for DB, r in results.items():
  n = len(r['scores_multi'])
  per_fold.append(pd.concat([pd.DataFrame({'Dataset': DB, 'Outer fold': np.arange(1, n + 1)}),
                             pd.DataFrame(r['chosen']), pd.DataFrame({'Inner best 3-class BA': r['inner_best']}),
                             r['m_bin'].add_prefix('Binary '), r['m_multi'].add_prefix('3-class ')], axis=1))
pd.concat(per_fold, ignore_index=True).to_csv(os.path.join(RESULTS_DIR, 'per_outer_fold.csv'), index=False)
pd.to_pickle({'results': results}, os.path.join(RESULTS_DIR, 'results.pkl'))
print('saved to', RESULTS_DIR, sorted(os.listdir(RESULTS_DIR)))